In [ ]:
#========================================
#       IMPORTER
#========================================

import requests #för att skicka HTTPS till API
import json # inbyggt verktyg för att arberta med json (JavaScriot Ibject Notation)
from datetime import datetime # importerar tid och datum

In [ ]:
#========================================
#       HJÄLPMETODER
#========================================
# Innehåller mindre funktioner som används
# på flera olika ställen i programmet.

def normalisera(text): #Gör texten till små bokstäver och tar bort onödiga mellanslag.
    """Gör texten enklare att jämföra."""
    return text.lower().strip()


def visa_kompetenser(kompetenser): #Tar en lista med kompetenser och gör om den till en text med kommatecken mellan varje kompetens.
    """Returnerar matchande kompetenser som text."""
    if kompetenser:
        return ", ".join(kompetenser)
    return "Inga"

In [ ]:
# ========================================
#      DATAKLASSER
# ========================================
# Innehåller basklasser och underklasser
# som hanterar data och information.


class MatchningsData:
    """Basklass för gemensam information."""

    def __init__(self):
        # Initierar listor för plats och kompetenser
        self.plats = []
        self.kompetenser = []


class Profil(MatchningsData):  # Barnklass till MatchningsData
    """Användarens profil."""

    def __init__(self):
        super().__init__()  # Anropar basklassen (MatchningsData)
        self.jobb = []

    def input_jobb(self):
        print("\n" + "=" * 40)

        try:
            # Hämtar önskade jobb från användaren och delar upp vid kommatecken
            jobb_input = input("Vad hade du velat jobba som? ").split(",")

            # Normaliserar och rensar bort tomma strängar från inmatningen
            jobb_input = [
                normalisera(jobb) for jobb in jobb_input if jobb.strip()
            ]

            # Lägger till de bearbetade jobben i profilens lista
            self.jobb.extend(jobb_input)

        except (KeyboardInterrupt, EOFError):
            # Fångar upp avbruten inmatning från användaren
            print("\nInmatningen avbröts.")

    def input_plats(self):
        print("\n" + "=" * 40)

        try:
            # Hämtar önskade platser från användaren och delar upp vid kommatecken
            plats_input = input("Vart hade du velat jobba? ").split(",")

            # Normaliserar och rensar bort tomma strängar från inmatningen
            plats_input = [
                normalisera(plats) for plats in plats_input if plats.strip()
            ]

            # Lägger till de bearbetade platserna i profilens lista
            self.plats.extend(plats_input)

        except (KeyboardInterrupt, EOFError):
            # Fångar upp avbruten inmatning från användaren
            print("\nInmatningen avbröts.")

    def input_kompetenser(self):
        print("\n" + "=" * 40)

        try:
            # Hämtar kompetenser från användaren och delar upp vid kommatecken
            kompetens_input = input("Vad har du för kompetenser? ").split(",")

            # Normaliserar och rensar bort tomma strängar från inmatningen
            kompetens_input = [
                normalisera(kompetens)
                for kompetens in kompetens_input
                if kompetens.strip()
            ]

            # Lägger till de bearbetade kompetenserna i profilens lista
            self.kompetenser.extend(kompetens_input)

        except (KeyboardInterrupt, EOFError):
            # Fångar upp avbruten inmatning från användaren
            print("\nInmatningen avbröts.")


class Jobbannons(MatchningsData):  # Barnklass till MatchningsData
    """Innehåller information om en jobbannons."""

    def __init__(self, jobb, plats, kompetenser, länk):
        super().__init__()  # Anropar basklassen (MatchningsData)

        # Tilldelar specifik information för jobbannonsen
        self.jobb = jobb
        self.plats = plats
        self.kompetenser = kompetenser
        self.länk = länk

In [ ]:
# ========================================
#      LOGIKKLASSER
# ========================================
# Innehåller klasser som utför beräkningar,
# matchningar och hanterar programmets logik.


class Matchare:
    """Matchar användarens profil mot jobbannonser."""

    def __init__(self, profil, arbetsmarknad):
        self.profil = profil
        self.arbetsmarknad = arbetsmarknad

    def match(self):
        matchningar = []

        # Går igenom alla jobbannonser
        for jobb in self.arbetsmarknad:

            jobb_matchar = False

            # Kontrollerar om jobbtiteln matchar
            for önskat_jobb in self.profil.jobb:

                if önskat_jobb in jobb.jobb:
                    jobb_matchar = True
                    break

            # Hoppa över jobbet om titeln inte matchar
            if not jobb_matchar:
                continue

            poäng = 0

            # Matchar önskad plats
            for plats in self.profil.plats:

                if plats in jobb.plats:
                    poäng += 2
                    break

            # Matchar användarens kompetenser
            for kompetens in self.profil.kompetenser:

                if kompetens in jobb.kompetenser:
                    poäng += 1

            # Ett jobb som matchar titeln får minst 1 poäng
            if poäng == 0:
                poäng = 1

            matchningar.append((jobb, poäng))

        # Högsta poäng visas först
        matchningar.sort(key=lambda x: x[1], reverse=True) #skickar info till .sort() (Titta p poängen och sortera dem baklänges)

        return matchningar

In [ ]:
# ========================================
#      API-FUNKTION
# ========================================
# Hämtar jobbannonser från JobTech Dev API,
# skapar objekt och sparar undan rådata till fil.


def hämta_jobb_från_api(profil):

    # Kontrollerar om profilen saknar jobb, och avbryter i så fall
    if not profil.jobb:
        print("Du måste ställa in din profil först.")
        return []

    url = "https://jobsearch.api.jobtechdev.se/search"

    jobbannonser = []
    alla_data = []

    try:

        # Söker efter varje önskat jobb i användarens profil
        for sökord in profil.jobb:

            print(f"\nSöker efter: {sökord}")

            params = {
                "q": sökord,
                "limit": 10
            }

            # Skickar en GET-förfrågan till API:et med angivna parametrar
            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            print("Statuskod:", response.status_code)

            # Kontrollerar om anropet misslyckades
            if response.status_code != 200:
                print("Kunde inte hämta jobbannonser.")
                continue

            # Omvandlar svaret till JSON-format och sparar rådatan
            data = response.json()
            alla_data.append(data)

            träffar = data.get("hits", [])

            print("Antal träffar:", len(träffar))

            # Skapar Jobbannons-objekt för varje träff
            for jobb in träffar:

                # Hämtar och normaliserar jobbets titel
                titel = normalisera(
                    jobb.get("headline") or ""
                )

                # Hämtar information om arbetsplats och kommun
                plats_info = jobb.get(
                    "workplace_address"
                ) or {}

                plats = normalisera(
                    plats_info.get("municipality") or ""
                )

                # Hämtar länken till annonsen
                länk = jobb.get(
                    "webpage_url"
                ) or ""

                # Hämtar och normaliserar jobbets beskrivningstext
                beskrivning = normalisera(
                    jobb.get(
                        "description", {}
                    ).get("text") or ""
                )

                matchade_kompetenser = []

                # Letar efter användarens kompetenser
                # i jobbannonsens beskrivningstext
                for kompetens in profil.kompetenser:

                    if kompetens in beskrivning:
                        matchade_kompetenser.append(
                            kompetens
                        )

                # Skapar ett nytt Jobbannons-objekt med den insamlade informationen
                annons = Jobbannons(
                    titel,
                    plats,
                    matchade_kompetenser,
                    länk
                )

                # Lägger till den färdiga annonsen i listan
                jobbannonser.append(annons)

        # ========================================
        #      SPARA API-DATA
        # ========================================

        try:
            # Öppnar filen för att skriva ner insamlad data med UTF-8 kodning
            with open(
                "jobbdata.json",
                "w",
                encoding="utf-8"
            ) as f:

                # Serialiserar listan till JSON-format och sparar den snyggt formaterad
                json.dump(
                    alla_data,
                    f,
                    ensure_ascii=False,
                    indent=4
                )

        except OSError as fel:
            # Fångar upp filrelaterade fel om det inte går att spara filen
            print("\nKunde inte spara jobbdata.")
            print(fel)

        # Returnerar listan med färdiga jobbannonser
        return jobbannonser

    except requests.exceptions.RequestException as fel:
        # Fångar upp nätverksrelaterade fel vid kommunikation med API:t
        print("\nEtt fel uppstod när API:t kontaktades.")
        print(fel)

        return []

    except json.JSONDecodeError:
        # Fångar upp fel om API-svaret inte är giltig JSON och inte kan tolkas
        print("\nKunde inte läsa svaret från API:t.")

        return []





In [ ]:
# ========================================
#     Historik
# ========================================
# Hanterar sparande och visning av
# tidigare sökningar och matchningar.


def spara_historik(profil, matchningar):

    try:
        # Försöker öppna och läsa befintlig historik från JSON-filen
        with open(
            "historik.json",
            "r",
            encoding="utf-8"
        ) as f:

            historik = json.load(f)

    except (FileNotFoundError, json.JSONDecodeError):
        # Om filen inte finns eller är tom/korrupt skapas en ny tom lista
        historik = []

    # Skapar en struktur för den aktuella sökningen
    ny_sökning = {
        "datum": datetime.now().strftime(
            "%Y-%m-%d %H:%M"
        ),
        "profil": {
            "jobb": profil.jobb,
            "plats": profil.plats,
            "kompetenser": profil.kompetenser
        },
        "matchningar": []
    }

    # Går igenom och sparar alla matchningar i sökningen
    for jobb, poäng in matchningar:

        ny_sökning["matchningar"].append({
            "jobb": jobb.jobb,
            "plats": jobb.plats,
            "poäng": poäng,
            "kompetenser": jobb.kompetenser,
            "länk": jobb.länk
        })

    # Lägger till den nya sökningen i den totala historiken
    historik.append(ny_sökning)

    try:
        # Sparar den uppdaterade historiken till JSON-filen
        with open(
            "historik.json",
            "w", # W Betder write
            encoding="utf-8" # Tillåter specialtecken som å, ä, ö
        ) as f:

            json.dump(
                historik,
                f,
                ensure_ascii=False,
                indent=4
            )

    except OSError as fel:
        # Fångar upp filrelaterade fel vid skrivning
        print("\nKunde inte spara historiken.")
        print(fel)
        return

    print("\nSökningen har sparats i historiken.")


def visa_historik():

    try:
        # Försöker läsa in historikfilen
        with open(
            "historik.json",
            "r",
            encoding="utf-8"
        ) as f:

            historik = json.load(f)

    except FileNotFoundError:
        # Meddelar om filen saknas
        print("\nDet finns ingen historik ännu.")
        return

    except json.JSONDecodeError:
        # Meddelar om filen är trasig/felaktig
        print("\nHistorikfilen kunde inte läsas.")
        return

    # Kontrollerar om listan är tom
    if not historik:
        print("\nDet finns ingen historik ännu.")
        return

    print("\n" + "=" * 40)
    print("             HISTORIK")
    print("=" * 40)

    # Går igenom tidigare sökningar och numrerar dem
    for nummer, sökning in enumerate(
        historik,
        start=1
    ):

        print(f"\nSökning {nummer}")
        print(f"Datum: {sökning['datum']}")

        print(
            f"Jobb: "
            f"{', '.join(sökning['profil']['jobb'])}"
        )

        print(
            f"Plats: "
            f"{', '.join(sökning['profil']['plats'])}"
        )

        print(
            f"Kompetenser: "
            f"{', '.join(sökning['profil']['kompetenser'])}"
        )

        print("\nMatchningar:")

        # Skriver ut information för varje matchat jobb i sökningen
        for jobb in sökning["matchningar"]:

            print(f"  - {jobb['jobb']}")
            print(f"    Plats: {jobb['plats']}")
            print(f"    Poäng: {jobb['poäng']}")

            print(
                "    Matchande kompetenser: "
                + visa_kompetenser(
                    jobb["kompetenser"]
                )
            )

            print(f"    Länk: {jobb['länk']}")

        print("-" * 40)

In [ ]:
# ========================================
#      VISA MATCHNINGAR
# ========================================
# Hämtar jobb från API, matchar dem mot profil,
# sparar resultatet i historiken och skriver ut allt.


def visa_matchningar(profil):

    print("\nHämtar aktuella jobbannonser...")

    # Hämtar jobbannonser baserat på användarens profil via API:et
    jobb_fran_api = hämta_jobb_från_api(profil)

    # Kontrollerar om inga jobb kunde hämtas
    if not jobb_fran_api:
        print("Inga jobbannonser kunde hämtas.")
        return

    # Initierar matcharen med profilen och de hämtade jobben
    matchare = Matchare(
        profil,
        jobb_fran_api
    )

    # Utför matchningen och beräknar poäng
    matchningar = matchare.match()

    # Kontrollerar om inga jobb matchade profilen
    if not matchningar:
        print("\nInga jobb matchade din profil.")
        return

    # Sparar sökningen och matchningarna i historiken
    spara_historik(
        profil,
        matchningar
    )

    print("\n" + "=" * 40)
    print("          DINA MATCHNINGAR")
    print("=" * 40)

    # Går igenom och skriver ut varje matchat jobb med dess detaljer
    for jobb, poäng in matchningar:

        print(f"\n- {jobb.jobb}")
        print(f"  Plats: {jobb.plats}")
        print(f"  Poäng: {poäng}")

        print(
            "  Matchande kompetenser: "
            + visa_kompetenser(
                jobb.kompetenser
            )
        )

        print(f"  Länk: {jobb.länk}")

In [ ]:
# ========================================
#      ANVÄNDARGRÄNSSNITT
# ========================================
# Skapar en profil-instans och hanterar
# programmets huvudmeny i terminalen.

profil = Profil()


def user_ui():

    # Startar en evighetsloop som håller menyn igång tills användaren väljer att avsluta
    while True:

        print("\n")
        print("=" * 40)
        print("         VÄLKOMMEN TILL AI-JOBBANALYS")
        print("=" * 40)

        print("1. Ställ in profil")
        print("2. Visa matchningar")
        print("3. Historik")
        print("4. Avsluta")

        print("=" * 40)

        try:
            # Tar emot användarens val i menyn
            svar = input(
                "Välj ett alternativ 1-4: "
            )

        except (KeyboardInterrupt, EOFError):
            # Fångar upp avbruten körning (t.ex. Ctrl+C)
            print("\nProgrammet avslutas.")
            break

In [10]:
# ========================================
#      ANVÄNDARGRÄNSSNITT
# ========================================
# Skapar en profil-instans och hanterar
# programmets huvudmeny i terminalen.

profil = Profil()


def user_ui():

    # Startar en evighetsloop som håller menyn igång tills användaren väljer att avsluta
    while True:

        print("\n")
        print("=" * 40)
        print("         VÄLKOMMEN TILL AI-JOBBANALYS")
        print("=" * 40)

        print("1. Ställ in profil")
        print("2. Visa matchningar")
        print("3. Historik")
        print("4. Avsluta")

        print("=" * 40)

        try:
            # Tar emot användarens val i menyn
            svar = input("Välj ett alternativ 1-4: ")

        except (KeyboardInterrupt, EOFError):
            # Fångar upp avbruten körning (t.ex. Ctrl+C)
            print("\nProgrammet avslutas.")
            break

        # ========================================
        #      PROFIL
        # ========================================

        if svar == "1":
            # Kör funktioner för att samla in information till profilen
            profil.input_jobb()
            profil.input_plats()
            profil.input_kompetenser()

            print("\nProfil ändrad!")

            # Skriver ut en sammanfattning av den nya profilen
            print(
                f"\nJobb: {profil.jobb}"
                f"\nPlats/Ort: {profil.plats}"
                f"\nKompetenser: {profil.kompetenser}"
            )

        # ========================================
        #      MATCHNINGAR
        # ========================================

        elif svar == "2":
            # Startar processen för att söka och visa matchande jobb
            visa_matchningar(profil)

        # ========================================
        #      HISTORIK
        # ========================================

        elif svar == "3":
            # Visar tidigare sparade sökningar och matchningar
            visa_historik()

        # ========================================
        #      AVSLUTA
        # ========================================

        elif svar == "4":
            # Avslutar loopen och programmet
            print("\nTack för denna gång!")
            break

        # ========================================
        #      FELAKTIGT VAL
        # ========================================

        else:
            # Hanterar ogiltig inmatning från användaren
            print("\nFelaktigt val.")
            print("Välj en siffra mellan 1 och 4.")


# ========================================
#      STARTA PROGRAMMET
# ========================================
# Anropar huvudfunktionen för att starta applikationen

user_ui()



         VÄLKOMMEN TILL AI-JOBBANALYS
1. Ställ in profil
2. Visa matchningar
3. Historik
4. Avsluta

Programmet avslutas.
